In [4]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import LabelEncoder

torch.manual_seed(42)
np.random.seed(42)

# Load - your file is in sample_data
df = pd.read_csv('/content/income.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

Shape: (48842, 15)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'class']
   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0

In [5]:
# Preprocessing for YOUR columns
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Use only 30000 rows as per task
df = df.head(30000)

# Handle? and NaN
df.replace('?', float('nan'), inplace=True)
df.replace('?', float('nan'), inplace=True)
df.dropna(inplace=True)

# Separate target
target_col = 'class'
y = df[target_col]
X = df.drop(target_col, axis=1)

# Encode categorical columns
cat_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
encoders = {}
for col in cat_cols:
    if col in X.columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le

# Encode target: <=50K=0, >50K=1
le_target = LabelEncoder()
y = le_target.fit_transform(y.astype(str))
print("Target classes:", le_target.classes_)

# Scale numeric
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("Final X shape:", X.shape, "y shape:", y.shape)

Target classes: ['<=50K' '>50K']
Final X shape: (27761, 14) y shape: (27761,)


In [6]:
from torch.utils.data import TensorDataset, DataLoader

# 80% train, 10% val, 10% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# Convert to tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.FloatTensor(y_val).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64)

Train: (22208, 14), Val: (2776, 14), Test: (2777, 14)


In [7]:
import torch.nn as nn

class IncomeClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

model = IncomeClassifier(X_train.shape[1])
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

IncomeClassifier(
  (network): Sequential(
    (0): Linear(in_features=14, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
)


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

epochs = 15
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_preds = []
    with torch.no_grad():
        for xb, yb in val_loader:
            val_preds.extend(model(xb).numpy())
    val_preds = (np.array(val_preds) > 0.5).astype(int)
    val_acc = accuracy_score(y_val, val_preds)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss/len(train_loader):.4f} - Val Acc: {val_acc:.4f}")

# Final Test
model.eval()
test_preds = []
with torch.no_grad():
    for xb, yb in test_loader:
        test_preds.extend(model(xb).numpy())
test_preds = (np.array(test_preds) > 0.5).astype(int)

print("\n--- FINAL TEST RESULTS ---")
print(f"Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Precision: {precision_score(y_test, test_preds):.4f}")
print(f"Recall: {recall_score(y_test, test_preds):.4f}")
print(f"F1: {f1_score(y_test, test_preds):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, test_preds))

Epoch 1/15 - Loss: 0.4038 - Val Acc: 0.8325
Epoch 2/15 - Loss: 0.3482 - Val Acc: 0.8451
Epoch 3/15 - Loss: 0.3419 - Val Acc: 0.8440
Epoch 4/15 - Loss: 0.3368 - Val Acc: 0.8444
Epoch 5/15 - Loss: 0.3354 - Val Acc: 0.8476
Epoch 6/15 - Loss: 0.3362 - Val Acc: 0.8480
Epoch 7/15 - Loss: 0.3343 - Val Acc: 0.8480
Epoch 8/15 - Loss: 0.3303 - Val Acc: 0.8440
Epoch 9/15 - Loss: 0.3304 - Val Acc: 0.8447
Epoch 10/15 - Loss: 0.3323 - Val Acc: 0.8444
Epoch 11/15 - Loss: 0.3326 - Val Acc: 0.8476
Epoch 12/15 - Loss: 0.3249 - Val Acc: 0.8476
Epoch 13/15 - Loss: 0.3263 - Val Acc: 0.8451
Epoch 14/15 - Loss: 0.3246 - Val Acc: 0.8458
Epoch 15/15 - Loss: 0.3298 - Val Acc: 0.8462

--- FINAL TEST RESULTS ---
Accuracy: 0.8470
Precision: 0.7422
Recall: 0.5646
F1: 0.6414
Confusion Matrix:
 [[1972  132]
 [ 293  380]]


In [9]:
torch.save(model.state_dict(), 'model.pth')
print("Saved model.pth - Download this + income.csv +.ipynb for GitHub")

Saved model.pth - Download this + income.csv +.ipynb for GitHub


In [10]:
!ls -lh /content

total 5.1M
-rw-r--r-- 1 root root 5.1M Sep  6 05:31 income.csv
-rw-r--r-- 1 root root  52K Sep  6 05:41 model.pth
drwxr-xr-x 1 root root 4.0K Aug 24 13:21 sample_data


In [11]:
from google.colab import files
files.download('model.pth')
files.download('income.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from google.colab import files
# This will download notebook name
!jupyter nbconvert --to notebook --output census_income_workshop.ipynb /content/Untitled10.ipynb 2>/dev/null || echo "try other"

This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePr